# 16 Model Evaluation And Deployment Basics

Evaluation metrics, cross-validation (time-series split), simple model persistence.

**Короткое объяснение:**

- Данные: синтетический ряд с трендом, сезонностью и шумом (200 точек).
- Цель: показать ключевые приёмы для изучаемой темы.

**ASCII-визуализация ряда:**

```
▄▄▅▅▄▂▁▁▁▃▅▅▅▄▃▁▁▂▄▅▅▆▅▂▂▁▂▄▅▆▆▄▃▁▂▃▅▆▆▆▅▃▃▃▄▅▇▆▅▄▃▃▄▄▆▇▇▆▄▃
```


In [ ]:
# Пример данных (первые 8 строк)
import pandas as pd
import numpy as np
from datetime import datetime
rng = pd.date_range(start="2020-01-01", periods=200, freq="D")
t = np.arange(len(rng))
seasonal = 10 * np.sin(2 * np.pi * t / 30)
trend = 0.05 * t
noise = np.random.normal(scale=2.0, size=len(rng))
series = 50 + trend + seasonal + noise
df = pd.DataFrame({'ds': rng, 'y': series})
df.head(8)


In [ ]:
# Быстрая визуализация (matplotlib)
import matplotlib.pyplot as plt
plt.figure(figsize=(8,3))
plt.plot(df['ds'], df['y'])
plt.title('Synthetic time series')
plt.xlabel('Date')
plt.ylabel('Value')
plt.tight_layout()
plt.show()


In [ ]:
# TimeSeriesSplit и метрики
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
tscv = TimeSeriesSplit(n_splits=3)
df_ml = df.copy()
for lag in (1,2,3):
    df_ml[f'lag_{lag}'] = df_ml['y'].shift(lag)
df_ml = df_ml.dropna()
X = df_ml[[c for c in df_ml.columns if c.startswith('lag_')]]
y = df_ml['y']
for train_idx, test_idx in tscv.split(X):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    m = LinearRegression().fit(Xtr, ytr)
    pred = m.predict(Xte)
    print('RMSE:', mean_squared_error(yte, pred, squared=False))
